In [13]:
import ROOT
%jsroot on

In [14]:
import ROOT
ROOT.gROOT.SetBatch(True)
ROOT.TH1.SetDefaultSumw2(True)
ROOT.gStyle.SetOptStat(0)

files = [
    "../geant_JP2_pt-hat55999_027_R0.2.root",
    "../geant_JP2_pt-hat55999_027_R0.6.root",
    "../geant_JP2_pt-hat55999_027_R0.2_old.root",
    "../geant_JP2_pt-hat55999_027_R0.6_old.root"
]
R = ['0.2', '0.6', '0.2 old', '0.6 old']

colors = [2000,2001,2002,2003]

can = ROOT.TCanvas("can", "Trigger efficiency", 800, 600)
leg = ROOT.TLegend(0.6, 0.7, 0.9, 0.88)
leg.SetBorderSize(0)
leg.SetFillStyle(0)
leg.SetTextSize(0.04)

effs = []

for i, f in enumerate(files):
    file = ROOT.TFile.Open(f)
    tree = file.Get("ResultTree")
    if not tree:
        print("Missing ResultTree in", f)
        continue

    # let ROOT create histograms via Draw
    tree.Draw(f"pt>>den_R{i}(100,0,100)", "", "goff")
    tree.Draw(f"pt>>num_R{i}(100,0,100)", "trigger_match_JP2", "goff")

    h_den = ROOT.gDirectory.Get(f"den_R{i}")
    h_num = ROOT.gDirectory.Get(f"num_R{i}")

    h_eff = h_num.Clone(f"eff_R{i}")
    h_eff.Divide(h_num, h_den, 1.0, 1.0, "B")

    h_eff.SetLineColor(colors[i])
    h_eff.SetLineWidth(2)
    h_eff.GetXaxis().SetTitle("p_{T} [GeV/c]")
    h_eff.GetYaxis().SetTitle("Trigger efficiency")
    h_eff.GetYaxis().SetRangeUser(0, 1.1)
    h_eff.SetDirectory(0)

    leg.AddEntry(h_eff, f"R = {R[i]}")
    effs.append(h_eff)
    file.Close()

if effs:
    effs[0].Draw("hist e")
    for h in effs[1:]:
        h.Draw("hist e same")
    leg.Draw()
    ROOT.gPad.SetGrid(1, 1)
    can.Update()
    # can.SaveAs("trigger_efficiency_comparison.pdf")


Warning in <TCanvas::Constructor>: Deleting canvas with same name: can


In [15]:
can.Draw()